In [1]:
import math
import seaborn as sns
import numpy as np
import scanpy as sc
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

## Loading & processing data

In [2]:
# Selecting the brain region
select_region = "Isocortex-1-IT-ET-Glut"

In [ ]:
# Loading AnnData object
base_path = Path("/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/")
expr_path = base_path / f"AnnData/WMB-10Xv3-{select_region}-raw-wmeta-filtered.h5ad"
adata = sc.read_h5ad(expr_path)
adata

In [4]:
gene_names_df = adata.var.copy()
adata.var.set_index("gene_symbol", inplace=True)
# adata.var_names_make_unique()

In [5]:
# Preprocess the data
# adata.raw = adata  # Store the raw data
sc.pp.normalize_total(adata, target_sum=1e4)  # Normalize counts per cell
sc.pp.log1p(adata)  # Log-transform the data
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True)  # Select highly variable genes

### DE

In [6]:
# Perform differential expression analysis
group_by="subclass"
sc.tl.rank_genes_groups(
    adata,
    groupby=group_by,
    method="wilcoxon",
    use_raw=False
)

In [7]:
# Extract the differential expression results
deg_results = pd.DataFrame({
    group: pd.DataFrame(adata.uns["rank_genes_groups"]["names"])[group]
    for group in adata.uns["rank_genes_groups"]["names"].dtype.names
})

# Save the dataframe to a CSV file
output_path = base_path / "outputs/DEG/" / f"WMB-10Xv3-{select_region}-DEG-{group_by}.csv"
deg_results.to_csv(output_path, index=False)

print(f"Differentially expressed genes saved to {output_path}")

Differentially expressed genes saved to /data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/outputs/DEG/WMB-10Xv3-Isocortex-1-IT-ET-Glut-DEG-subclass.csv


In [8]:
deg_results.head()

,001 CLA-EPd-CTX Car3 Glut,002 IT EP-CLA Glut,003 L5/6 IT TPE-ENT Glut,004 L6 IT CTX Glut,005 L5 IT CTX Glut,006 L4/5 IT CTX Glut,007 L2/3 IT CTX Glut,020 L2/3 IT RSP Glut,021 L4 RSP-ACA Glut,022 L5 ET CTX Glut
0,Synpr,Sulf1,Fxyd6,Cdh9,Il1rapl2,Rorb,Rasgrf2,Tshz2,Cbln1,Bcl11b
1,Nr4a2,Nnat,Cnr1,Sema3e,Sulf2,Kcnk2,Rgs6,Slc17a6,Parm1,Fezf2
2,Rgs12,Chst11,Timp2,Bmp3,Tmsb10,Nrsn1,Calb1,Zfpm2,Ccdc136,Parm1
3,Gnb4,Nfia,C1ql3,Igsf21,Deptor,Cnih3,Gsg1l,Htr2c,Ntng1,Hcn1
4,B3gat2,Gria1,Gap43,Galnt14,Grik3,Kcnh5,Dgkb,Unc13c,Nrip3,Rab3c


In [9]:
# Saving processed AnnData object
base_path = Path("/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/")
save_path = base_path / f"processed/WMB-10Xv3-{select_region}-raw-wmeta-filtered-DEG-{group_by}.h5ad"
sc.write(save_path, adata)